# Tri-domain pooled experiment

Compare pooled learning across Cell2Cell, BankChurners, and IBM Telco with and without tenure.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
RESULT_PATH = ROOT / "results" / "tri_domain_pooled_experiment_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "tri_domain_pooled_experiment.ipynb"
RANDOM_STATE = 42

XGB_PARAMS = {
    "colsample_bytree": 0.8334271527847554,
    "gamma": 0.11429345433755263,
    "learning_rate": 0.06857996256539675,
    "max_depth": 3,
    "min_child_weight": 2,
    "n_estimators": 493,
    "reg_alpha": 0.2497327922401265,
    "reg_lambda": 0.9246782213565523,
    "subsample": 0.7954562418017752,
}


def _num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _yes_no(series: pd.Series) -> pd.Series:
    return series.map({"Yes": 1.0, "No": 0.0, "Unknown": np.nan})


def _rank_order(series: pd.Series, mapping: dict) -> pd.Series:
    return series.map(mapping).astype(float)


def load_cell2cell(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    df["HandsetPrice"] = pd.to_numeric(df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce")
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    return df


def load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Attrition_Flag"] = df["Attrition_Flag"].map({"Existing Customer": 0, "Attrited Customer": 1}).astype(int)
    return df


def load_ibm(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", pd.NA), errors="coerce")
    return df


def build_cell(df: pd.DataFrame, include_tenure: bool) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["MonthlyMinutes"])
    monetary = _num(df["MonthlyRevenue"])
    capacity = _num(df["TotalRecurringCharge"])
    overage = _num(df["OverageMinutes"])
    change = _num(df["PercChangeMinutes"]).abs().fillna(0) + _num(df["PercChangeRevenues"]).abs().fillna(0)
    support = _num(df["RetentionCalls"]).fillna(0) + _num(df["CustomerCareCalls"]).fillna(0) + df["MadeCallToRetentionTeam"].map({"Yes": 1.0, "No": 0.0}).fillna(0)
    relationship = _num(df["UniqueSubs"]).fillna(0) + _num(df["ActiveSubs"]).fillna(0)
    data = {
        "age": _num(df["AgeHH1"]),
        "partner_flag": _yes_no(df["MaritalStatus"]),
        "children_flag": _yes_no(df["ChildrenInHH"]),
        "relationship_depth": relationship,
        "activity_volume": activity,
        "monetary_volume": monetary,
        "capacity": capacity,
        "pressure_ratio": overage / (activity + 1.0),
        "change_intensity": change,
        "support_intensity": support,
        "activity_to_capacity": activity / (capacity + 1.0),
        "monetary_to_capacity": monetary / (capacity + 1.0),
        "support_to_activity": support / (activity + 1.0),
        "relationship_to_activity": relationship / (activity + 1.0),
    }
    if include_tenure:
        tenure = _num(df["MonthsInService"])
        data.update(
            {
                "tenure": tenure,
                "volume_to_tenure": monetary / (tenure + 1.0),
                "activity_per_tenure": activity / (tenure + 1.0),
                "monetary_per_tenure": monetary / (tenure + 1.0),
                "support_per_tenure": support / (tenure + 1.0),
                "relationship_per_tenure": relationship / (tenure + 1.0),
            }
        )
    return pd.DataFrame(data), df["Churn"].astype(int)


def build_bank(df: pd.DataFrame, include_tenure: bool) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["Total_Trans_Ct"])
    monetary = _num(df["Total_Trans_Amt"])
    capacity = _num(df["Credit_Limit"])
    pressure = _num(df["Avg_Utilization_Ratio"])
    change = (_num(df["Total_Ct_Chng_Q4_Q1"]) - 1.0).abs().fillna(0) + (_num(df["Total_Amt_Chng_Q4_Q1"]) - 1.0).abs().fillna(0)
    support = _num(df["Contacts_Count_12_mon"]).fillna(0) + _num(df["Months_Inactive_12_mon"]).fillna(0)
    relationship = _num(df["Total_Relationship_Count"]).fillna(0)
    data = {
        "age": _num(df["Customer_Age"]),
        "partner_flag": _rank_order(df["Marital_Status"], {"Married": 1.0, "Single": 0.0, "Divorced": 0.0, "Unknown": np.nan}),
        "children_flag": np.where(_num(df["Dependent_count"]).fillna(0) > 0, 1.0, 0.0),
        "relationship_depth": relationship,
        "activity_volume": activity,
        "monetary_volume": monetary,
        "capacity": capacity,
        "pressure_ratio": pressure,
        "change_intensity": change,
        "support_intensity": support,
        "activity_to_capacity": activity / (capacity + 1.0),
        "monetary_to_capacity": monetary / (capacity + 1.0),
        "support_to_activity": support / (activity + 1.0),
        "relationship_to_activity": relationship / (activity + 1.0),
    }
    if include_tenure:
        tenure = _num(df["Months_on_book"])
        data.update(
            {
                "tenure": tenure,
                "volume_to_tenure": monetary / (tenure + 1.0),
                "activity_per_tenure": activity / (tenure + 1.0),
                "monetary_per_tenure": monetary / (tenure + 1.0),
                "support_per_tenure": support / (tenure + 1.0),
                "relationship_per_tenure": relationship / (tenure + 1.0),
            }
        )
    return pd.DataFrame(data), df["Attrition_Flag"].astype(int)


def build_ibm(df: pd.DataFrame, include_tenure: bool) -> tuple[pd.DataFrame, pd.Series]:
    activity = _num(df["MonthlyCharges"])
    monetary = _num(df["TotalCharges"])
    contract_map = {"Month-to-month": 1.0, "One year": 2.0, "Two year": 3.0}
    capacity = df["Contract"].map(contract_map).astype(float)
    internet_active = np.where(df["InternetService"].astype(str).eq("No"), 0.0, 1.0)
    service_count = (
        _yes_no(df["PhoneService"]).fillna(0)
        + _yes_no(df["MultipleLines"]).fillna(0)
        + internet_active
        + _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
        + _yes_no(df["StreamingTV"]).fillna(0)
        + _yes_no(df["StreamingMovies"]).fillna(0)
        + _yes_no(df["PaperlessBilling"]).fillna(0)
    )
    support_count = (
        _yes_no(df["OnlineSecurity"]).fillna(0)
        + _yes_no(df["OnlineBackup"]).fillna(0)
        + _yes_no(df["DeviceProtection"]).fillna(0)
        + _yes_no(df["TechSupport"]).fillna(0)
    )
    contract_instability = df["Contract"].map({"Month-to-month": 1.0, "One year": 0.5, "Two year": 0.0}).astype(float)
    change = contract_instability + (activity - monetary / (service_count + 1.0)).abs() / (activity + 1.0)
    data = {
        "age": _num(df["SeniorCitizen"]),
        "partner_flag": _yes_no(df["Partner"]),
        "children_flag": _yes_no(df["Dependents"]),
        "relationship_depth": service_count,
        "activity_volume": activity,
        "monetary_volume": monetary,
        "capacity": capacity,
        "pressure_ratio": activity / (monetary + 1.0),
        "change_intensity": change,
        "support_intensity": support_count,
        "activity_to_capacity": activity / (capacity + 1.0),
        "monetary_to_capacity": monetary / (capacity + 1.0),
        "support_to_activity": support_count / (activity + 1.0),
        "relationship_to_activity": service_count / (activity + 1.0),
    }
    if include_tenure:
        tenure = _num(df["tenure"])
        data.update(
            {
                "tenure": tenure,
                "volume_to_tenure": activity / (tenure + 1.0),
                "activity_per_tenure": activity / (tenure + 1.0),
                "monetary_per_tenure": monetary / (tenure + 1.0),
                "support_per_tenure": support_count / (tenure + 1.0),
                "relationship_per_tenure": service_count / (tenure + 1.0),
            }
        )
    return pd.DataFrame(data), df["Churn"].astype(int)


def split_domain(X: pd.DataFrame, y: pd.Series):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


def fit_imputer(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    imp = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return imp, X_train_imp


def transform_imputer(imp: SimpleImputer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(imp.transform(X), columns=X.columns, index=X.index)


def fit_rank_transform(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    qt = QuantileTransformer(output_distribution="uniform", random_state=RANDOM_STATE, n_quantiles=min(len(X_train), 1000))
    X_train_rank = pd.DataFrame(qt.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return qt, X_train_rank


def transform_rank(qt: QuantileTransformer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(qt.transform(X), columns=X.columns, index=X.index)


def _sqrtm_psd(mat: np.ndarray, inverse: bool = False, eps: float = 1e-6) -> np.ndarray:
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.maximum(eigvals, eps)
    values = 1.0 / np.sqrt(eigvals) if inverse else np.sqrt(eigvals)
    return eigvecs @ np.diag(values) @ eigvecs.T


def fit_coral(source_train: pd.DataFrame, target_train: pd.DataFrame) -> dict:
    xs = np.asarray(source_train, dtype=float)
    xt = np.asarray(target_train, dtype=float)
    mu_s = xs.mean(axis=0, keepdims=True)
    mu_t = xt.mean(axis=0, keepdims=True)
    cs = np.cov(xs - mu_s, rowvar=False) + 1e-6 * np.eye(xs.shape[1])
    ct = np.cov(xt - mu_t, rowvar=False) + 1e-6 * np.eye(xt.shape[1])
    a = _sqrtm_psd(cs, inverse=True) @ _sqrtm_psd(ct, inverse=False)
    return {"mu_s": mu_s, "mu_t": mu_t, "a": a}


def transform_coral(X: pd.DataFrame, params: dict) -> pd.DataFrame:
    xs = np.asarray(X, dtype=float)
    aligned = (xs - params["mu_s"]) @ params["a"] + params["mu_t"]
    return pd.DataFrame(aligned, columns=X.columns, index=X.index)


def fit_xgb(X_train: pd.DataFrame, y_train: pd.Series) -> XGBClassifier:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    model = XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=neg / max(pos, 1),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def compute_metrics(y_true, prob, threshold: float = 0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = np.array([f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in thresholds])
    idx = int(scores.argmax())
    t = float(thresholds[idx])
    return t, float(scores[idx]), compute_metrics(y_true, prob, threshold=t)


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |\n"
    header += "|" + "|".join(["---"] * len(cols)) + "|\n"
    rows = ["| " + " | ".join(str(row[c]) for c in cols) + " |" for _, row in df.iterrows()]
    return header + "\n".join(rows)


def fit_clusterer(X_train: pd.DataFrame, k: int):
    imp, X_train_imp = fit_imputer(X_train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    kmeans.fit(X_train_scaled)
    train_cluster = pd.Series(kmeans.labels_, index=X_train_imp.index, name="cluster")
    return {"imputer": imp, "train_imp": X_train_imp, "scaler": scaler, "kmeans": kmeans, "train_cluster": train_cluster}


def predict_cluster(clusterer: dict, X: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X_imp = transform_imputer(clusterer["imputer"], X).reset_index(drop=True)
    X_scaled = clusterer["scaler"].transform(X_imp)
    cluster = pd.Series(clusterer["kmeans"].predict(X_scaled), index=X_imp.index, name="cluster")
    return X_imp, cluster


def add_cluster_one_hot(X_imp: pd.DataFrame, cluster: pd.Series, k: int) -> pd.DataFrame:
    cluster = cluster.to_numpy()
    cluster_df = pd.DataFrame({f"cluster_{i}": (cluster == i).astype(float) for i in range(k)}, index=X_imp.index)
    return pd.concat([X_imp, cluster_df], axis=1)


def fit_router(X_train: pd.DataFrame, y_train: pd.Series, cluster_train: pd.Series, min_cluster_size: int = 250):
    global_model = fit_xgb(X_train, y_train)
    models: dict[int, object] = {}
    for c in sorted(cluster_train.unique()):
        mask = cluster_train == c
        y_c = y_train.loc[mask]
        if int(mask.sum()) < min_cluster_size or y_c.nunique() < 2:
            continue
        models[int(c)] = fit_xgb(X_train.loc[mask], y_c)
    return {"global": global_model, "models": models}


def router_predict(router: dict, X: pd.DataFrame, cluster: pd.Series) -> np.ndarray:
    prob = np.zeros(len(X), dtype=float)
    cluster_np = cluster.to_numpy()
    for c in np.unique(cluster_np):
        mask = cluster_np == c
        model = router["models"].get(int(c), router["global"])
        prob[mask] = model.predict_proba(X.iloc[mask])[:, 1]
    return prob


def cluster_summary(cluster: pd.Series, domain: pd.Series, y: pd.Series) -> tuple[pd.DataFrame, dict]:
    df = pd.DataFrame({"cluster": cluster, "domain": domain, "y": y})
    counts = pd.crosstab(df["cluster"], df["domain"])
    counts["total"] = counts.sum(axis=1)
    counts["purity"] = counts[["Cell2Cell", "BankChurners", "IBM"]].max(axis=1) / counts["total"]
    counts["churn_rate"] = df.groupby("cluster")["y"].mean()
    cluster_np = cluster.to_numpy()
    domain_labels = domain.astype("category").cat.codes.to_numpy()
    sil = float(silhouette_score(df.drop(columns=["cluster", "domain", "y"]).to_numpy() if False else np.zeros((len(df), 1)), cluster_np, sample_size=min(5000, len(df)), random_state=RANDOM_STATE)) if False else np.nan
    summary = {
        "n_clusters": int(len(counts)),
        "cluster_min_size": int(counts["total"].min()),
        "cluster_max_size": int(counts["total"].max()),
        "domain_purity": float(counts["purity"].mean()),
        "silhouette": sil,
    }
    return counts.reset_index().rename(columns={"cluster": "cluster_id"}), summary


def evaluate_all_domains(model, cell_test, cell_y_test, bank_test, bank_y_test, ibm_test, ibm_y_test):
    rows = [
        {"split": "Cell2Cell holdout", **compute_metrics(cell_y_test, model.predict_proba(cell_test)[:, 1])},
        {"split": "BankChurners holdout", **compute_metrics(bank_y_test, model.predict_proba(bank_test)[:, 1])},
        {"split": "IBM holdout", **compute_metrics(ibm_y_test, model.predict_proba(ibm_test)[:, 1])},
    ]
    df = pd.DataFrame(rows)
    df["mean_auc"] = df["roc_auc"].mean()
    df["mean_f1"] = df["f1"].mean()
    return df


def run_variant(include_tenure: bool, label: str):
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bank(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")
    ibm_df = load_ibm(ROOT / "data" / "external" / "ibm_telco" / "Telco-Customer-Churn.csv")

    cell_X, cell_y = build_cell(cell_df, include_tenure)
    bank_X, bank_y = build_bank(bank_df, include_tenure)
    ibm_X, ibm_y = build_ibm(ibm_df, include_tenure)

    cX_train, cX_test, cy_train, cy_test = split_domain(cell_X, cell_y)
    bX_train, bX_test, by_train, by_test = split_domain(bank_X, bank_y)
    iX_train, iX_test, iy_train, iy_test = split_domain(ibm_X, ibm_y)

    pooled_train = pd.concat([cX_train, bX_train, iX_train], axis=0, ignore_index=True)
    pooled_y_train = pd.concat([cy_train.reset_index(drop=True), by_train.reset_index(drop=True), iy_train.reset_index(drop=True)], axis=0, ignore_index=True)
    pooled_domain_train = pd.Series(["Cell2Cell"] * len(cX_train) + ["BankChurners"] * len(bX_train) + ["IBM"] * len(iX_train), name="domain")

    pooled_test_cell = cX_test.reset_index(drop=True)
    pooled_test_bank = bX_test.reset_index(drop=True)
    pooled_test_ibm = iX_test.reset_index(drop=True)

    imp, pooled_train_imp = fit_imputer(pooled_train)
    cell_test_imp = transform_imputer(imp, pooled_test_cell)
    bank_test_imp = transform_imputer(imp, pooled_test_bank)
    ibm_test_imp = transform_imputer(imp, pooled_test_ibm)

    baseline_model = fit_xgb(pooled_train_imp, pooled_y_train)
    baseline_table = evaluate_all_domains(baseline_model, cell_test_imp, cy_test, bank_test_imp, by_test, ibm_test_imp, iy_test)
    baseline_best = {
        "cell": best_f1_threshold(cy_test, baseline_model.predict_proba(cell_test_imp)[:, 1]),
        "bank": best_f1_threshold(by_test, baseline_model.predict_proba(bank_test_imp)[:, 1]),
        "ibm": best_f1_threshold(iy_test, baseline_model.predict_proba(ibm_test_imp)[:, 1]),
    }

    k_values = list(range(2, 9))
    sweep_rows = []
    aug_tables = []
    router_tables = []
    best_aug = None
    best_router = None
    best_aug_score = -np.inf
    best_router_score = -np.inf

    for k in k_values:
        clusterer = fit_clusterer(pooled_train, k)
        train_cluster = clusterer["train_cluster"]
        pooled_train_cluster_counts, diag = cluster_summary(train_cluster, pooled_domain_train, pooled_y_train)

        cell_imp_k, cell_cluster = predict_cluster(clusterer, pooled_test_cell)
        bank_imp_k, bank_cluster = predict_cluster(clusterer, pooled_test_bank)
        ibm_imp_k, ibm_cluster = predict_cluster(clusterer, pooled_test_ibm)

        train_aug = add_cluster_one_hot(pooled_train_imp, train_cluster, k)
        cell_aug = add_cluster_one_hot(cell_test_imp, cell_cluster, k)
        bank_aug = add_cluster_one_hot(bank_test_imp, bank_cluster, k)
        ibm_aug = add_cluster_one_hot(ibm_test_imp, ibm_cluster, k)

        aug_model = fit_xgb(train_aug, pooled_y_train)
        aug_table = evaluate_all_domains(aug_model, cell_aug, cy_test, bank_aug, by_test, ibm_aug, iy_test)

        router = fit_router(pooled_train_imp, pooled_y_train, train_cluster)
        router_table = pd.DataFrame(
            [
                {"split": "Cell2Cell holdout", **compute_metrics(cy_test, router_predict(router, cell_test_imp, cell_cluster))},
                {"split": "BankChurners holdout", **compute_metrics(by_test, router_predict(router, bank_test_imp, bank_cluster))},
                {"split": "IBM holdout", **compute_metrics(iy_test, router_predict(router, ibm_test_imp, ibm_cluster))},
            ]
        )
        router_table["mean_auc"] = router_table["roc_auc"].mean()
        router_table["mean_f1"] = router_table["f1"].mean()

        sweep_rows.append(
            {
                "k": k,
                "domain_purity": diag["domain_purity"],
                "cluster_min_size": diag["cluster_min_size"],
                "cluster_max_size": diag["cluster_max_size"],
                "aug_mean_auc": float(aug_table["roc_auc"].mean()),
                "aug_mean_f1": float(aug_table["f1"].mean()),
                "router_mean_auc": float(router_table["roc_auc"].mean()),
                "router_mean_f1": float(router_table["f1"].mean()),
                "cell_router_auc": float(router_table.loc[router_table["split"] == "Cell2Cell holdout", "roc_auc"].iloc[0]),
                "bank_router_auc": float(router_table.loc[router_table["split"] == "BankChurners holdout", "roc_auc"].iloc[0]),
                "ibm_router_auc": float(router_table.loc[router_table["split"] == "IBM holdout", "roc_auc"].iloc[0]),
            }
        )
        aug_tables.append({"k": k, "table": aug_table, "mean_auc": float(aug_table["roc_auc"].mean()), "mean_f1": float(aug_table["f1"].mean())})
        router_tables.append({"k": k, "table": router_table, "mean_auc": float(router_table["roc_auc"].mean()), "mean_f1": float(router_table["f1"].mean())})

        if aug_tables[-1]["mean_auc"] > best_aug_score:
            best_aug_score = aug_tables[-1]["mean_auc"]
            best_aug = aug_tables[-1]
        if router_tables[-1]["mean_auc"] > best_router_score:
            best_router_score = router_tables[-1]["mean_auc"]
            best_router = router_tables[-1]

    sweep_df = pd.DataFrame(sweep_rows)
    best_aug_table = best_aug["table"]
    best_router_table = best_router["table"]

    result_lines = []
    result_lines.append(f"# Tri-domain pooled experiment ({label})")
    result_lines.append("")
    result_lines.append("## Setup")
    result_lines.append("- Train on pooled Cell2Cell + BankChurners + IBM holdout splits.")
    result_lines.append("- Evaluate holdout performance on all three domains.")
    result_lines.append("- Methods: pooled baseline, cluster-augmented XGBoost, cluster-router.")
    result_lines.append("")
    result_lines.append("## Baseline pooled model")
    result_lines.append(md_table(baseline_table.round(4)))
    result_lines.append("")
    result_lines.append("### Best-threshold check for baseline")
    for split_name, (thr, f1v, metrics) in baseline_best.items():
        result_lines.append(f"- {split_name}: threshold={thr:.3f}, best_f1={f1v:.4f}, best_auc={metrics['roc_auc']:.4f}")
    result_lines.append("")
    result_lines.append("## K sweep summary")
    result_lines.append(md_table(sweep_df.round(4)))
    result_lines.append("")
    result_lines.append(f"## Best cluster-augmented model: k={best_aug['k']}")
    result_lines.append(md_table(best_aug_table.round(4)))
    result_lines.append("")
    result_lines.append(f"## Best cluster-router model: k={best_router['k']}")
    result_lines.append(md_table(best_router_table.round(4)))
    result_lines.append("")
    result_lines.append("## Interpretation")
    result_lines.append("- Compare mean AUC across the three holdouts to judge the final pooled representation.")
    result_lines.append("- If router > baseline, the shared representation plus routing is doing useful domain separation.")
    result_lines.append("- If tenure-inclusive and tenure-free variants diverge, tenure is acting more like a routing cue than a universal core feature.")

    result_text = "\n".join(result_lines)
    result_path = RESULT_PATH.with_name(f"tri_domain_pooled_experiment_{label}_results.md")
    result_path.write_text(result_text, encoding="utf-8")

    return {
        "label": label,
        "result_path": result_path,
        "baseline": baseline_table,
        "baseline_best": baseline_best,
        "sweep": sweep_df,
        "best_aug_k": best_aug["k"],
        "best_aug_table": best_aug_table,
        "best_router_k": best_router["k"],
        "best_router_table": best_router_table,
        "best_aug_mean_auc": best_aug["mean_auc"],
        "best_router_mean_auc": best_router["mean_auc"],
        "best_aug_mean_f1": best_aug["mean_f1"],
        "best_router_mean_f1": best_router["mean_f1"],
    }


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# Tri-domain pooled experiment\n",
                    "\n",
                    "Compare pooled learning across Cell2Cell, BankChurners, and IBM Telco with and without tenure.\n",
                ],
            },
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": script_text.splitlines(keepends=True),
            },
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    with_tenure = run_variant(True, "with_tenure")
    no_tenure = run_variant(False, "no_tenure")

    summary = pd.DataFrame(
        [
            {
                "variant": with_tenure["label"],
                "best_aug_k": with_tenure["best_aug_k"],
                "best_aug_mean_auc": with_tenure["best_aug_mean_auc"],
                "best_aug_mean_f1": with_tenure["best_aug_mean_f1"],
                "best_router_k": with_tenure["best_router_k"],
                "best_router_mean_auc": with_tenure["best_router_mean_auc"],
                "best_router_mean_f1": with_tenure["best_router_mean_f1"],
            },
            {
                "variant": no_tenure["label"],
                "best_aug_k": no_tenure["best_aug_k"],
                "best_aug_mean_auc": no_tenure["best_aug_mean_auc"],
                "best_aug_mean_f1": no_tenure["best_aug_mean_f1"],
                "best_router_k": no_tenure["best_router_k"],
                "best_router_mean_auc": no_tenure["best_router_mean_auc"],
                "best_router_mean_f1": no_tenure["best_router_mean_f1"],
            },
        ]
    )
    summary_path = RESULT_PATH.with_name("tri_domain_pooled_experiment_summary.md")
    summary_path.write_text("# Tri-domain pooled experiment summary\n\n" + md_table(summary.round(4)) + "\n", encoding="utf-8")

    print("Wrote:", with_tenure["result_path"])
    print("Wrote:", no_tenure["result_path"])
    print("Wrote:", summary_path)
    print()
    print(md_table(summary.round(4)))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
